In [ ]:
#| default_exp match

# Match

> Nomenclature reconciliation: fuzzy matching, lookup tables, and external API resolution.

## Core

In [ ]:
#| export
from pathlib import Path
from dataclasses import dataclass
from math import modf
from fastcore.all import *
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests
from jellyfish import levenshtein_distance, jaro_winkler_similarity
from typing import List, Dict, Callable, Tuple, Optional, Union
from marisco.configs import cache_path, lut_fname

In [ ]:
#| export
def uniq_across_dfs(dfs:Dict[str,pd.DataFrame],  # Dict of group DataFrames
                    col:str,                      # Column to extract unique values from
                   )->list:                      # Unique values across all group DataFrames
    "Unique column values across all group DataFrames."
    return list(set().union(*(df[col].unique() for df in dfs.values() if col in df.columns)))

For instance:

In [ ]:
dfs = {'SEAWATER': pd.DataFrame({'NUCLIDE': ['cs137', 'cs134_137_tot', 'cs134_137_tot']}),
       'BIOTA': pd.DataFrame({'NUCLIDE': ['cs137', 'cs134', 'cs134_137_tot']}),
       'SEDIMENT': pd.DataFrame({'NUCLIDE': ['cs134_137_tot', 'cs134_137_tot', 'cs134_137_tot']})}

test_eq(set(uniq_across_dfs(dfs, 'NUCLIDE')), {'cs134', 'cs137', 'cs134_137_tot'})

What if the column name is not in one of the dataframe?

In [ ]:
dfs = {'SEAWATER': pd.DataFrame({'NUCLIDE': ['cs137', 'cs134_137_tot', 'cs134_137_tot']}),
       'BIOTA': pd.DataFrame({'NUCLIDE': ['cs137', 'cs134', 'cs134_137_tot']}),
       'SEDIMENT': pd.DataFrame({'NONUCLIDE': ['cs134_137_tot', 'cs134_137_tot', 'cs134_137_tot']})}

test_eq(set(uniq_across_dfs(dfs, 'NUCLIDE')), {'cs134', 'cs137', 'cs134_137_tot'})

When a data provider doesn't supply its own nomenclature, we can derive one from the data itself. `lut_from` wraps `uniq_across_dfs` in the format that `Remapper` expects.

In [ ]:
#| export
def lut_from(dfs: Dict[str, pd.DataFrame],
             col: str,
             incl_nchars: bool=False
            ) -> pd.DataFrame:
    "Build a source lookup table from unique values across all DataFrames."
    vals = sorted(uniq_across_dfs(dfs, col))
    df = pd.DataFrame(vals, columns=['value'])
    if incl_nchars: df['n_chars'] = df['value'].str.len()
    return df

In [ ]:
# lut_from example
df_lut = lut_from(dfs, 'NUCLIDE')
test_eq(list(df_lut['value']), sorted(['cs134', 'cs137', 'cs134_137_tot']))


In [ ]:
# lut_from with incl_nchars
df_lut = lut_from(dfs, 'NUCLIDE', incl_nchars=True)
test_eq(list(df_lut['n_chars']), [5, 13, 5])

# single group
single = {'SEAWATER': pd.DataFrame({'NUCLIDE': ['cs137', 'cs137']})}
test_eq(uniq_across_dfs(single, 'NUCLIDE'), ['cs137'])


## MIFA: Nomenclature Reconciliation

A four-step pattern for reconciling provider nomenclature against MARIS reference lookups.

This is a four-step pattern for reconciling provider nomenclature against MARIS reference lookups. It is designed for the reality that **nomenclature reconciliation is edgy**, it can't be fully automated, and reliably gets the last mile right only with domain expertise in the loop.

The idea is:

1. **M**atch: let the computer do brute-force fuzzy matching (it's good at that)
2. **I**nspect: look at the non-exact matches (score > 0) to find problems
3. **F**ix: apply expert overrides for cases the fuzzy match got wrong
4. **A**pply: use the result as a callable lookup

Handler authors should follow this pattern whenever they need to align provider nomenclature (species names, nuclide codes, units, etc.) to MARIS identifiers. The functions below give you the building blocks; the "Inspect" step is a manual review of the borderline matches.

## MIFA: Nomenclature Reconciliation

A four-step pattern for reconciling provider nomenclature against MARIS reference lookups.

This is a four-step pattern for reconciling provider nomenclature against MARIS reference lookups. It is designed for the reality that **nomenclature reconciliation is edgy**, it can't be fully automated, and reliably gets the last mile right only with domain expertise in the loop.

The idea is:

1. **M**atch: let the computer do brute-force fuzzy matching (it's good at that)
2. **I**nspect: look at the non-exact matches (score > 0) to find problems
3. **F**ix: apply expert overrides for cases the fuzzy match got wrong
4. **A**pply: use the result as a callable lookup

Handler authors should follow this pattern whenever they need to align provider nomenclature (species names, nuclide codes, units, etc.) to MARIS identifiers. The functions below give you the building blocks; the "Inspect" step is a manual review of the borderline matches.

In [ ]:
#| export
def fuzzy_merge(left: pd.DataFrame,
                right: pd.DataFrame,
                left_on: str='value',
                right_on: str='name',
                dist_fn: Callable=levenshtein_distance,
               ) -> pd.DataFrame:
    "For each row in left, find closest row in right by dist_fn."
    rows = []
    for _, lrow in left.iterrows():
        best_d = float('inf')
        best_rrow = None
        for _, rrow in right.iterrows():
            d = dist_fn(lrow[left_on], rrow[right_on])
            if d < best_d:
                best_d = d
                best_rrow = rrow
        rows.append({**lrow.to_dict(), **best_rrow.to_dict(), 'score': best_d})
    return pd.DataFrame(rows)

Test `fuzzy_merge` exact matches, near-matches, and custom distance functions:

In [ ]:
# fuzzy_merge: exact matches get score 0
left = pd.DataFrame({'value': ['cs137', 'k40']})
right = pd.DataFrame({'name': ['cs137', 'k40', 'sr90'], 'maris_id': [1, 2, 3]})
merged = fuzzy_merge(left, right, left_on='value', right_on='name')
test_eq(list(merged['score']), [0, 0])
test_eq(list(merged['maris_id']), [1, 2])

# fuzzy_merge: near-matches get a non-zero score
left = pd.DataFrame({'value': ['cs-137', 'cs134_137']})
merged = fuzzy_merge(left, right, left_on='value', right_on='name')
# 'cs-137' → 'cs137' (dist 1), 'cs134_137' → 'cs137' (dist 5)
test_eq(merged.loc[merged['value'] == 'cs-137', 'score'].iloc[0], 1)
test_eq(merged.loc[merged['value'] == 'cs134_137', 'score'].iloc[0], 4)

In [ ]:
# fuzzy_merge: custom distance function
left = pd.DataFrame({'value': ['cs137', 'k40']})
merged_jw = fuzzy_merge(left, right, left_on='value', right_on='name',
                         dist_fn=lambda a, b: 1 - jaro_winkler_similarity(a, b))
test_eq(list(merged_jw['maris_id']), [1, 2])

For more details on jellyfish distance/similarity functions, see the [official documentation](https://www.jpt.sh/projects/jellyfish/functions/).

In [ ]:
#| export
def fix_lut(merged: pd.DataFrame,
            overrides: dict,
            maris: pd.DataFrame,
            left_on: str,
            right_on: str,
            id_col: str,
           ) -> pd.DataFrame:
    "Replace matched entries with expert overrides by name."
    merged = merged.copy()
    for src_val, target_name in overrides.items():
        mid = maris.loc[maris[right_on] == target_name, id_col].iloc[0]
        mask = merged[left_on] == src_val
        merged.loc[mask, [id_col, right_on, 'score']] = [mid, target_name, 0]
    return merged

`fix_lut` replaces fuzzy-matched entries with expert overrides and resets their score to 0:

In [ ]:
# fix_lut: override a fuzzy match with the correct MARIS name
maris = pd.DataFrame({'name': ['cs137', 'k40', 'cs134_137_tot'], 'maris_id': [1, 2, 33]})
left = pd.DataFrame({'value': ['cs134_137']})
merged = fuzzy_merge(left, maris, left_on='value', right_on='name')
# cs134_137 matched to cs137 (score 4) — wrong!
test_eq(merged['maris_id'].iloc[0], 1)
test_eq(merged['score'].iloc[0], 4)
print(merged)

       value   name  maris_id  score
0  cs134_137  cs137         1      4


In [ ]:
# Fix it with an expert override
overrides = {'cs134_137': 'cs134_137_tot'}
fixed = fix_lut(merged, overrides, maris,
                left_on='value', right_on='name', id_col='maris_id')
test_eq(fixed['maris_id'].iloc[0], 33)
test_eq(fixed['score'].iloc[0], 0)
print(fixed)

       value           name  maris_id  score
0  cs134_137  cs134_137_tot        33      0


In [ ]:
# Empty overrides: no changes
fixed2 = fix_lut(merged, {}, maris,
                 left_on='value', right_on='name', id_col='maris_id')
test_eq(fixed2['maris_id'].iloc[0], 1)
test_eq(fixed2['score'].iloc[0], 4)
print(fixed)

       value           name  maris_id  score
0  cs134_137  cs134_137_tot        33      0


In [ ]:
#| export
class Lut:
    "Callable lookup: maps a source value to a MARIS id."
    def __init__(self, merged: pd.DataFrame,
                 key_col: str,
                 val_col: str='maris_id'):
        store_attr()

    def __call__(self, x):
        return self.merged.loc[self.merged[self.key_col] == x, self.val_col].iloc[0]

Let's test `Lut` — a simple callable that maps source values to MARIS identifiers:

In [ ]:
# Lut: callable lookup maps source value to MARIS id
merged = pd.DataFrame({'value': ['cs137', 'k40'], 'maris_id': [1, 2]})
lut = Lut(merged, key_col='value', val_col='maris_id')
test_eq(lut('cs137'), 1)
test_eq(lut('k40'), 2)

# Lut with default column names
merged2 = pd.DataFrame({'source': ['sr90'], 'maris_id': [3]})
lut2 = Lut(merged2, key_col='source')
test_eq(lut2('sr90'), 3)


### Usage examples

In [ ]:
# Case 1 — Provider has explicit nomenclature (like HELCOM RUBIN_NAME.csv)
provider_df = pd.DataFrame({
    'RUBIN': ['GADU MOR', 'FUCU VES', 'MYTI EDU'],
    'SCIENTIFIC NAME': ['GADUS MORHUA', 'FUCUS VESICULOSUS', 'MYTILUS EDULIS'],
})

maris_species = pd.DataFrame({
    'species_id': [11, 22, 33],
    'species_name': ['Gadus morhua', 'Fucus vesiculosus', 'Mytilus edulis'],
})

overwrite_cache = False
path = cache_path() / 'species_helcom.pkl'
if path.exists() and not overwrite_cache:
    merged = pd.read_pickle(path)
else:
    merged = fuzzy_merge(provider_df, maris_species,
                         left_on='SCIENTIFIC NAME', right_on='species_name')
    merged.to_pickle(path)

merged.query('score > 0')  # inspect non-exact matches

merged = fix_lut(merged, {}, maris_species,
                 left_on='SCIENTIFIC NAME', right_on='species_name', id_col='species_id')
lut = Lut(merged, key_col='SCIENTIFIC NAME', val_col='species_id')

print(lut('GADUS MORHUA'))  # 11

11


### Case 2 — Provider without explicit nomenclature

Here the provider does not supply a nomenclature lookup table. We infer the unique values directly from the data using `lut_from`, then follow the same MIFA pipeline.


In [ ]:
# Case 2 — Provider without explicit nomenclature (use lut_from to infer from data)
provider_data = {
    'SEAWATER': pd.DataFrame({'NUCLIDE': ['cs137', 'cs134', 'cs137', 'k40']}),
    'BIOTA':    pd.DataFrame({'NUCLIDE': ['cs137', 'k40', 'sr90', 'cs134_137_tot']}),
}

# Inspect: build a LUT from the data itself
provider_lut = lut_from(provider_data, 'NUCLIDE')

maris_nuclides = pd.DataFrame({
    'maris_id': [1, 2, 3, 33],
    'name': ['cs137', 'k40', 'sr90', 'cs134_137_tot'],
})

# Match: brute-force fuzzy matching
merged = fuzzy_merge(provider_lut, maris_nuclides, left_on='value', right_on='name')
# Uncomment to inspect borderline matches: merged.query('score > 0')

# Fix: override anything the fuzzy match got wrong
overrides = {'cs134_137_tot': 'cs134_137_tot'}
fixed = fix_lut(merged, overrides, maris_nuclides,
                left_on='value', right_on='name', id_col='maris_id')

# Apply: use as callable lookup
lut = Lut(fixed, key_col='value', val_col='maris_id')
test_eq(lut('cs137'), 1)
test_eq(lut('cs134_137_tot'), 33)

## WoRMS
The [World Register of Marine Species (WorMS)](https://www.marinespecies.org)  REST API for fuzzy species name matching.

In [ ]:
#| export
def match_worms(name: str):
    "Lookup `name` in WoRMS (fuzzy match)."
    url = 'https://www.marinespecies.org/rest/AphiaRecordsByMatchNames'
    params = {
        'scientificnames[]': [name],
        'marine_only': 'true'
    }
    headers = {'accept': 'application/json'}
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        return -1

For instance:

In [ ]:
#|eval: false
match_worms('Aristeus antennatus')

[[{'AphiaID': 107083,
   'url': 'https://www.marinespecies.org/aphia.php?p=taxdetails&id=107083',
   'scientificname': 'Aristeus antennatus',
   'authority': '(Risso, 1816)',
   'status': 'accepted',
   'unacceptreason': None,
   'taxonRankID': 220,
   'rank': 'Species',
   'valid_AphiaID': 107083,
   'valid_name': 'Aristeus antennatus',
   'valid_authority': '(Risso, 1816)',
   'parentNameUsageID': 106807,
   'originalNameUsageID': 342458,
   'kingdom': 'Animalia',
   'phylum': 'Arthropoda',
   'class': 'Malacostraca',
   'order': 'Decapoda',
   'family': 'Aristeidae',
   'genus': 'Aristeus',
   'citation': 'DecaNet eds. (2026). DecaNet. Aristeus antennatus (Risso, 1816). Accessed through: World Register of Marine Species at: https://www.marinespecies.org/aphia.php?p=taxdetails&id=107083 on 2026-06-19',
   'lsid': 'urn:lsid:marinespecies.org:taxname:107083',
   'isMarine': 1,
   'isBrackish': 0,
   'isFreshwater': 0,
   'isTerrestrial': 0,
   'isExtinct': 0,
   'match_type': 'exact',


## Deprecated

Kept for backward compatibility. Use the MIFA pattern instead.

In [ ]:
@dataclass
class Match:
    "Match between a data provider name and a MARIS lookup table."
    matched_id: int
    matched_maris_name: str
    source_name: str
    match_score: int

In [ ]:
def match_maris_lut(
    lut: Union[str, pd.DataFrame, Path],
    data_provider_name: str,
    maris_id: str,
    maris_name: str,
    dist_fn: Callable = levenshtein_distance,
    nresults: int = 10
) -> pd.DataFrame:
    "Fuzzy matching data provider and MARIS lookup tables."
    if isinstance(lut, str) or isinstance(lut, Path):
        df = pd.read_excel(lut)
    elif isinstance(lut, pd.DataFrame):
        df = lut
    else:
        raise ValueError("lut must be either a file path or a DataFrame")

    df = df.dropna(subset=[maris_name])
    df = df.astype({maris_id: 'int'})
    df['score'] = df[maris_name].str.lower().apply(lambda x: dist_fn(data_provider_name.lower(), x))
    df = df.sort_values(by='score', ascending=True)[:nresults]
    return df[[maris_id, maris_name, 'score']]

In [ ]:
class Remapper():
    "Remap a data provider lookup table to a MARIS lookup table using fuzzy matching."
    def __init__(self,
                 provider_lut_df: pd.DataFrame,
                 maris_col_id: str,
                 maris_col_name: str,
                 provider_col_to_match: str,
                 provider_col_key: str,
                 fname_cache: str,
                 maris_lut_fn: Union[Callable, pd.DataFrame]=None,
                 maris_lut_key: str=None,
                 ):
        store_attr()
        self.cache_file = cache_path() / fname_cache
        if maris_lut_key is not None:
            self.maris_lut = pd.read_excel(lut_fname(maris_lut_key))
        elif callable(maris_lut_fn):
            self.maris_lut = maris_lut_fn()
        else:
            self.maris_lut = maris_lut_fn
        self.lut = {}

    def generate_lookup_table(self,
                              fixes={},
                              as_df=True,
                              overwrite=True):
        self.fixes = fixes
        self.as_df = as_df
        if overwrite or not self.cache_file.exists():
            self._create_lookup_table()
            save_pickle(self.cache_file, self.lut)
        else:
            self.lut = load_pickle(self.cache_file)
        return self._format_output()

    def _create_lookup_table(self):
        df = self.provider_lut_df
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
            self._process_row(row)

    def _process_row(self, row):
        value_to_match = row[self.provider_col_to_match]
        if isinstance(value_to_match, str):
            name_to_match = self.fixes.get(value_to_match, value_to_match)
            result = match_maris_lut(self.maris_lut, name_to_match, self.maris_col_id, self.maris_col_name).iloc[0]
            match = Match(result[self.maris_col_id], result[self.maris_col_name],
                          value_to_match, result['score'])
            self.lut[row[self.provider_col_key]] = match
        else:
            self.lut[row[self.provider_col_key]] = Match(-1, "Unknown", value_to_match, 0)

    def select_match(self, match_score_threshold:int=1, verbose:bool=False):
        if verbose:
            matched_len = len([v for v in self.lut.values() if v.match_score < match_score_threshold])
            print(f"{matched_len} entries matched the criteria, while {len(self.lut) - matched_len} entries had a match score of {match_score_threshold} or higher.")
        self.lut = {k: v for k, v in self.lut.items() if v.match_score >= match_score_threshold}
        return self._format_output()

    def _format_output(self):
        if not self.as_df: return self.lut
        df_lut = pd.DataFrame.from_dict(self.lut, orient='index',
                                        columns=['matched_maris_name', 'source_name', 'match_score'])
        df_lut.index.name = 'source_key'
        return df_lut.sort_values(by='match_score', ascending=False)